# slm-finetune: QLoRA text-to-SQL on a free GPU

Fine-tunes **Qwen2.5-1.5B-Instruct** with QLoRA (Unsloth) to write SQLite for one fixed e-commerce schema, then
scores base vs fine-tuned with execution accuracy and exports a GGUF for Ollama.

**Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU T4 x2** (one is used), Internet **on**.

Expected time on a T4 with ~3k training pairs: about 25–40 min of training, 10 min of eval, 10 min of GGUF export.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code and data

Option A (recommended): push the repo to GitHub first (it includes `data/generated/` once you have generated it), then set `REPO_URL`.
Option B: zip the repo folder on your PC, upload it in the Files panel, and set `REPO_URL = ""`.

In [ ]:
import os, pathlib
REPO_URL = "https://github.com/<you>/slm-finetune"   # <- change me, or "" for an uploaded zip
WORK = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle") else "/content")
os.chdir(WORK)
if REPO_URL and not (WORK / "slm-finetune").exists():
    !git clone --depth 1 {REPO_URL} slm-finetune
elif not REPO_URL and not (WORK / "slm-finetune").exists():
    !unzip -q slm-finetune.zip -d {WORK}
os.chdir(WORK / "slm-finetune")
!ls

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q -r requirements.txt

In [ ]:
!python -m sqlbench.schema          # builds db/shop.db deterministically (same bytes as on your PC)
!wc -l data/generated/train.jsonl data/generated/val.jsonl data/generated/test_synth.jsonl data/gold_test.jsonl

No `data/generated/train.jsonl`? Generate it here with Claude: add `ANTHROPIC_API_KEY` under Colab **Secrets** (key icon), then run the
next cell. It prints the running cost; ~3k pairs with the default teacher costs a few dollars. Skip this if the files exist.

In [ ]:
GENERATE = False   # set True to generate data in the notebook
if GENERATE:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    !python data/generate.py --backend anthropic:claude-opus-5-5 --target 3000 --workers 8

## 2. Train (QLoRA)

Logs go to `outputs/qwen2.5-1.5b-sql/train_log.csv`. For Weights & Biases add `--report-to wandb` after `wandb login`.

In [ ]:
!python train/train.py --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --lr 2e-4 --lora-r 16 --lora-alpha 32     --output-dir outputs/qwen2.5-1.5b-sql

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("outputs/qwen2.5-1.5b-sql/train_log.csv")
ax = log.dropna(subset=["loss"]).plot(x="step", y="loss", figsize=(8, 4))
log.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=ax, marker="o")
plt.title("QLoRA training"); plt.savefig("outputs/qwen2.5-1.5b-sql/loss.png", dpi=120); plt.show()

## 3. Evaluate base vs fine-tuned (same prompt, same test sets)

`GPU_HOURLY` is the self-hosted cost model: seconds per request × hourly GPU price. Set it to what a T4 costs on the provider you
would deploy on, and keep it the same for every local run so the comparison is fair.

In [ ]:
GPU_HOURLY = 0.35   # USD/hour for a T4-class GPU; change to your provider's price
BASE = "unsloth/Qwen2.5-1.5B-Instruct"
ADAPTER = "outputs/qwen2.5-1.5b-sql/lora_adapter"
for test in ["gold", "synth"]:
    !python eval/run_eval.py --backend hf:{BASE} --load-in-4bit --test-set {test} --gpu-hourly-usd {GPU_HOURLY}         --hardware "Colab T4 (4-bit)" --label "Qwen2.5-1.5B-Instruct (base)" --run-name qwen1.5b-base__{test}
    !python eval/run_eval.py --backend hf:{BASE} --adapter {ADAPTER} --load-in-4bit --test-set {test} --gpu-hourly-usd {GPU_HOURLY}         --hardware "Colab T4 (4-bit)" --label "Qwen2.5-1.5B QLoRA (ours)" --run-name qwen1.5b-qlora__{test}
!python eval/report.py

## 4. Export GGUF for Ollama

In [ ]:
!python serve/export_gguf.py --adapter outputs/qwen2.5-1.5b-sql/lora_adapter --quant q4_k_m
!ls -lh serve/gguf serve/Modelfile

## 5. Take the results home

Download the zip, unzip it into your local `slm-finetune` folder, then on your PC:
`ollama create sql-qwen -f serve/Modelfile` and run the local eval (see README). Results JSONs go in `results/`.

In [ ]:
!zip -qr slm-artifacts.zip results outputs/qwen2.5-1.5b-sql/lora_adapter outputs/qwen2.5-1.5b-sql/*.json     outputs/qwen2.5-1.5b-sql/*.csv outputs/qwen2.5-1.5b-sql/*.png serve/gguf serve/Modelfile
!ls -lh slm-artifacts.zip
try:
    from google.colab import files; files.download("slm-artifacts.zip")
except ImportError:
    print("Kaggle: download slm-artifacts.zip from the Output panel")

## 6. (Optional) Push to Hugging Face

Add `HF_TOKEN` (write scope) to Secrets. The first command is a dry run; review it, then run with `--yes`.

In [ ]:
PUSH = False
if PUSH:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    !python scripts/push_to_hf.py --user <hf-username>
    # !python scripts/push_to_hf.py --user <hf-username> --yes